In [1]:
import os
import ast
import cv2
import numpy as np
import pandas as pd

In [2]:
def polygon_to_mask_pixel_space(coords, width, height):
    """
    coords: list[(x,y)] in ORIGINAL IMAGE PIXELS
    width, height: original image size
    """
    mask = np.zeros((height, width), dtype=np.uint8)

    if not coords or len(coords) < 3:
        return mask

    pts = np.array(coords, dtype=np.int32)

    # Ensure closed polygon
    if not np.array_equal(pts[0], pts[-1]):
        pts = np.vstack([pts, pts[0]])

    cv2.fillPoly(mask, [pts], 1)
    return mask

In [3]:
def calculate_iou(mask1, mask2):
    mask1 = mask1.astype(bool)
    mask2 = mask2.astype(bool)

    intersection = np.logical_and(mask1, mask2).sum()
    union = np.logical_or(mask1, mask2).sum()

    if union == 0:
        return 1.0

    return intersection / union

In [ ]:
# Paths
TEST_CSV = "test_with_path.csv"
PRED_CSV = "poly_predictions.csv"
MASK_DIR = "masks_full_json"

# Load CSVs
test_df = pd.read_csv(TEST_CSV)
pred_df = pd.read_csv(PRED_CSV)

# Build lookup for width / height
size_lookup = (
    test_df
    .set_index("ID")[["width", "height"]]
    .to_dict(orient="index")
)

ious = []

In [ ]:
# IoU computation loop
for _, row in pred_df.iterrows():
    img_id = row["ID"]

    # Default IoU if anything fails
    iou = 0.0

    try:
        # width & height
        if img_id not in size_lookup:
            ious.append(iou)
            continue

        width = int(size_lookup[img_id]["width"])
        height = int(size_lookup[img_id]["height"])

        # parse polygon
        poly_raw = row["Predicted_Polygon"]

        if pd.isna(poly_raw):
            ious.append(iou)
            continue

        # Safely parse string to list
        coords = ast.literal_eval(poly_raw)

        # predicted mask
        pred_mask = polygon_to_mask_pixel_space(
            coords=coords,
            width=width,
            height=height
        )

        # ground truth mask
        mask_path = os.path.join(MASK_DIR, f"{img_id}.png")

        if not os.path.exists(mask_path):
            ious.append(iou)
            continue

        gt_mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        gt_mask = (gt_mask > 0).astype(np.uint8)

        # resize safety (rare but important)
        if gt_mask.shape != pred_mask.shape:
            gt_mask = cv2.resize(
                gt_mask,
                (width, height),
                interpolation=cv2.INTER_NEAREST
            )

        # IoU
        iou = calculate_iou(pred_mask, gt_mask)

    except Exception as e:
        pass

    ious.append(iou)

# Insert IoU column after ID
pred_df.insert(
    loc=pred_df.columns.get_loc("ID") + 1,
    column="IoU",
    value=ious
)

# Average IoU
mean_iou = float(np.mean(ious))

print(f"Average IoU: {mean_iou:.6f}")

pred_df.to_csv("poly_predictions_with_iou.csv", index=False)


Average IoU: 0.974954
